# Step 21: The Ultimate Risk-Adaptive Portfolio Simulation

## 🎯 Objective:
Menggabungkan **AI Multi-Asset**, **Graph Theory**, dan **Markowitz Optimization** ke dalam satu simulasi backtest dengan **4 tingkat konservatisme (Threshold)**.

In [6]:
# Instalasi dependensi jika belum ada
!pip install PyPortfolioOpt cvxopt

Defaulting to user installation because normal site-packages is not writeable
  Using cached pyportfolioopt-1.5.6-py3-none-any.whl.metadata (22 kB)
  Using cached cvxopt-1.3.2-cp313-cp313-win_amd64.whl.metadata (1.4 kB)
  Using cached cvxpy-1.8.1-cp313-cp313-win_amd64.whl.metadata (9.8 kB)
  Using cached ecos-2.0.14.tar.gz (142 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached plotly-5.24.1-py3-none-any.whl.metadata (7.3 kB)
  Using cached osqp-1.1.0-cp313-cp313-win_amd64.whl.metadata (2.1 kB)
  Using cached clarabel-0.11.1-cp39-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached scs-3.2.11-cp313-cp313-win_amd64.whl.metadata (2.8 kB)
  Using cached highspy-1.12.0-cp313-cp313-win_amd64.whl.met

  error: subprocess-exited-with-error
  
  × Building wheel for ecos (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [12 lines of output]
      C:\Users\USER\AppData\Local\Temp\pip-build-env-ghuj5uk6\overlay\Lib\site-packages\setuptools\_distutils\dist.py:289: UserWarning: Unknown distribution option: 'tests_require'
        warnings.warn(msg)
      running bdist_wheel
      running build
      running build_py
      creating build\lib.win-amd64-cpython-313\ecos
      copying src\ecos\ecos.py -> build\lib.win-amd64-cpython-313\ecos
      copying src\ecos\version.py -> build\lib.win-amd64-cpython-313\ecos
      copying src\ecos\__init__.py -> build\lib.win-amd64-cpython-313\ecos
      running build_ext
      building '_ecos' extension
      error: Microsoft Visual C++ 14.0 or greater is required. Get it with "Microsoft C++ Build Tools": https://visualstudio.microsoft.com/visual-cpp-build-tools/
      [end of output]
  
  note: This error originates from a subprocess, a

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from pypfopt.efficient_frontier import EfficientFrontier
from pypfopt import risk_models, expected_returns
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')

print("Ultimate Simulation Environment Ready!")

ModuleNotFoundError: No module named 'pypfopt'

## 1. Data Preparation & Feature Engineering

In [ ]:
df = pd.read_excel('dataset_2023_2025.xlsx', index_col=0, parse_dates=True)
returns = df.pct_change().fillna(0)
crypto_assets = [c for c in df.columns if c != 'PAXG-USD']

scaler = StandardScaler()
pca = PCA(n_components=5)
pca_factors = pca.fit_transform(scaler.fit_transform(returns[crypto_assets]))

X_hybrid = pd.DataFrame(pca_factors, index=returns.index, columns=[f'PC{i+1}' for i in range(5)])
X_hybrid['BTC_Vol_20'] = returns['BTC-USD'].rolling(20).std()

def calc_density(ret_df, window=20):
    rolling_corr = ret_df.rolling(window).corr()
    v = len(ret_df.columns)
    max_e = v * (v - 1) / 2
    densities = []
    for date in ret_df.index[window-1:]:
        corr = rolling_corr.loc[date].fillna(0)
        edges = ((corr > 0.5).astype(int).values.sum() - v) / 2
        densities.append(edges / max_e if max_e > 0 else 0)
    return pd.Series(densities, index=ret_df.index[window-1:])

X_hybrid['Graph_Density'] = calc_density(returns[crypto_assets])
X_hybrid['Target'] = np.where(returns['BTC-USD'].shift(-1) > 0.005, 1, 0)
X_hybrid = X_hybrid.dropna()

train_mask = X_hybrid.index.year < 2025
test_mask = X_hybrid.index.year == 2025

X_train, y_train = X_hybrid[train_mask].drop(columns=['Target']), X_hybrid.loc[train_mask, 'Target']
X_test, y_test = X_hybrid[test_mask].drop(columns=['Target']), X_hybrid.loc[test_mask, 'Target']

model = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
probs = model.predict_proba(X_test)[:, 1]
print("Model Ready.")

## 2. Optimization Logic

In [ ]:
def get_weights(current_returns):
    corr = current_returns.corr().fillna(0)
    G = nx.Graph()
    for i, a1 in enumerate(corr.columns):
        for j, a2 in enumerate(corr.columns):
            if i < j and corr.iloc[i, j] > 0.5:
                G.add_edge(a1, a2)
    
    mis = nx.maximal_independent_set(G)
    # Add isolated assets
    all_crypto = set(current_returns.columns)
    selected = list(set(mis) | (all_crypto - set(G.nodes())))
    
    mu = expected_returns.mean_historical_return(current_returns[selected])
    S = risk_models.sample_cov(current_returns[selected])
    try:
        ef = EfficientFrontier(mu, S)
        w = ef.max_sharpe()
        return ef.clean_weights()
    except: return None

## 3. Backtest

In [ ]:
thresholds = {'Std': 0.5, 'Cts': 0.65, 'Cons': 0.8, 'Para': 0.9}
results = pd.DataFrame(index=X_test.index)

for name, t in thresholds.items():
    rets = []
    for i, date in enumerate(X_test.index):
        if probs[i] >= t:
            lookback = returns.loc[:date].iloc[-60:][crypto_assets]
            w = get_weights(lookback)
            if w: rets.append(sum(returns.loc[date, a] * v for a, v in w.items()))
            else: rets.append(returns.loc[date, 'PAXG-USD'])
        else:
            rets.append(returns.loc[date, 'PAXG-USD'])
    results[name] = rets

results['BTC'] = returns.loc[X_test.index, 'BTC-USD']
cum_rets = (1 + results).cumprod()
cum_rets.plot(figsize=(12, 6), title='Final Performance Comparison')
plt.show()